In [1]:
# ============================================================
# 1. 导入库与参数设置
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from scipy.io import savemat, loadmat
from torch.utils.data import Dataset


# -------------------- 文件路径 --------------------

# 原始 Excel 数据
INPUT_XLSX = Path("./data/1、2、3.xlsx")

# 筛选后的 Excel 数据
FILTERED_XLSX = Path("./filtered_voc_step2.xlsx")

# 最终供模型使用的 MAT 数据
OUTPUT_MAT = Path("./data/voc_dataset_1+2_vs_3.mat")

# 记录每个特征是否通过筛选
FEATURE_LOG_CSV = Path("./data/preprocessing_feature_log.csv")


# -------------------- 预处理参数 --------------------

# False：保留 Unknown，用于复现原始实验
# True：删除所有名字恰好为 Unknown 的代谢物
REMOVE_UNKNOWN = False

# Step 1：保留平均丰度高于第 40 百分位数的特征
ABUNDANCE_PERCENTILE = 40

# Step 2：保留 IQR 不低于第 25 百分位数的特征
IQR_PERCENTILE = 25

# 原始标签映射：
# 类别 1、2 合并为 0
# 类别 3 作为 1
LABEL_MAP = {
    1: 0,
    2: 0,
    3: 1,
}


print("预处理配置")
print("-" * 60)
print(f"输入文件：{INPUT_XLSX}")
print(f"是否删除 Unknown：{REMOVE_UNKNOWN}")
print(f"丰度筛选百分位数：P{ABUNDANCE_PERCENTILE}")
print(f"IQR 筛选百分位数：P{IQR_PERCENTILE}")
print(f"标签映射：{LABEL_MAP}")

预处理配置
------------------------------------------------------------
输入文件：data/1、2、3.xlsx
是否删除 Unknown：False
丰度筛选百分位数：P40
IQR 筛选百分位数：P25
标签映射：{1: 0, 2: 0, 3: 1}


In [2]:
# ============================================================
# 2. 读取并检查原始 Excel 数据
# ============================================================

if not INPUT_XLSX.exists():
    raise FileNotFoundError(
        f"找不到原始数据文件：{INPUT_XLSX.resolve()}"
    )

df = pd.read_excel(INPUT_XLSX)

if df.shape[0] < 2:
    raise ValueError(
        "Excel 至少需要两行：第一行代谢物名称，后续行为样本。"
    )

if df.shape[1] < 2:
    raise ValueError(
        "Excel 至少需要两列：第一列标签，后续列为 VOC 特征。"
    )


# Excel 列名作为特征 ID
all_feature_id = np.asarray(
    [str(value).strip() for value in df.columns[1:]],
    dtype=object,
)

# Excel 第一行作为代谢物名称
feature_name_series = df.iloc[0, 1:].copy()
feature_name_series = feature_name_series.fillna("Unknown")
all_feature_name = np.asarray(
    [str(value).strip() for value in feature_name_series],
    dtype=object,
)

# 从第二行开始读取 VOC 数值
feature_df = df.iloc[1:, 1:].apply(
    pd.to_numeric,
    errors="coerce",
)

# 从第二行开始读取原始标签
class_series = pd.to_numeric(
    df.iloc[1:, 0],
    errors="coerce",
)


# -------------------- 检查标签 --------------------

if class_series.isna().any():
    bad_rows = class_series[class_series.isna()].index.tolist()

    raise ValueError(
        "标签列中存在空值或非数字内容。"
        f"异常 Excel 行索引：{bad_rows[:10]}"
    )

classes = class_series.to_numpy(dtype=np.int64)

unknown_labels = sorted(
    set(classes.tolist()) - set(LABEL_MAP.keys())
)

if unknown_labels:
    raise ValueError(
        f"发现 LABEL_MAP 中没有定义的标签：{unknown_labels}"
    )


# -------------------- 检查 VOC 数值 --------------------

missing_count = int(feature_df.isna().sum().sum())

if missing_count > 0:
    bad_positions = np.argwhere(feature_df.isna().to_numpy())

    example_positions = [
        {
            "sample_row": int(row),
            "feature_column": int(col),
        }
        for row, col in bad_positions[:10]
    ]

    raise ValueError(
        f"VOC 数据中存在 {missing_count} 个空值或非数字内容。\n"
        f"前几个异常位置：{example_positions}"
    )

data_raw = feature_df.to_numpy(dtype=np.float64)

if not np.isfinite(data_raw).all():
    raise ValueError(
        "VOC 数据包含 NaN 或无穷大，请先检查原始 Excel。"
    )

if np.any(data_raw < 0):
    negative_count = int(np.sum(data_raw < 0))

    raise ValueError(
        f"VOC 数据中存在 {negative_count} 个负数。"
        "当前代码使用 log1p，原始丰度应当为非负数。"
    )


# -------------------- 检查维度 --------------------

if data_raw.shape[1] != len(all_feature_id):
    raise ValueError(
        "VOC 数据列数与特征 ID 数量不一致。"
    )

if data_raw.shape[1] != len(all_feature_name):
    raise ValueError(
        "VOC 数据列数与代谢物名称数量不一致。"
    )

if data_raw.shape[0] != len(classes):
    raise ValueError(
        "样本数量与标签数量不一致。"
    )


print("原始数据读取完成")
print("-" * 60)
print(f"样本数：{data_raw.shape[0]}")
print(f"原始特征数：{data_raw.shape[1]}")
print(f"数据最小值：{data_raw.min():.6f}")
print(f"数据最大值：{data_raw.max():.6f}")
print(f"标签及数量：{np.unique(classes, return_counts=True)}")
print(f"前 5 个特征 ID：{all_feature_id[:5].tolist()}")
print(f"前 5 个代谢物名称：{all_feature_name[:5].tolist()}")

原始数据读取完成
------------------------------------------------------------
样本数：159
原始特征数：1734
数据最小值：0.000000
数据最大值：80663080.000000
标签及数量：(array([1, 2, 3]), array([53, 53, 53]))
前 5 个特征 ID：['0', '1', '2', '3', '4']
前 5 个代谢物名称：['Unknown', '2-Butenal, (Z)-', 'Unknown', 'Unknown', '4,5-Dihydrooxazole-5-one, 4-[4-acetoxy-3-methoxybenzylidene]-2-phenyl-']


In [3]:
# ============================================================
# 3. 可选：删除 Unknown 特征
# ============================================================

normalized_feature_names = np.asarray(
    [str(name).strip().lower() for name in all_feature_name],
    dtype=object,
)

unknown_mask = normalized_feature_names == "unknown"

print(f"名字为 Unknown 的特征数：{int(unknown_mask.sum())}")


if REMOVE_UNKNOWN:
    keep_mask = ~unknown_mask

    if keep_mask.sum() == 0:
        raise ValueError(
            "删除 Unknown 后没有剩余特征。"
        )

    data_raw = data_raw[:, keep_mask]
    all_feature_id = all_feature_id[keep_mask]
    all_feature_name = all_feature_name[keep_mask]

    print("已删除 Unknown 特征。")
else:
    print("当前保留 Unknown 特征，用于复现原始结果。")


print(f"Unknown 处理后的特征数：{data_raw.shape[1]}")

名字为 Unknown 的特征数：746
当前保留 Unknown 特征，用于复现原始结果。
Unknown 处理后的特征数：1734


In [4]:
# ============================================================
# 4. Step 1：平均丰度筛选
# ============================================================

mean_data = np.mean(data_raw, axis=0)

threshold_abundance = np.percentile(
    mean_data,
    ABUNDANCE_PERCENTILE,
)

# 保留平均丰度严格高于 P40 的特征
mask_step1 = mean_data > threshold_abundance

if mask_step1.sum() == 0:
    raise ValueError(
        "丰度筛选后没有剩余特征，请降低 ABUNDANCE_PERCENTILE。"
    )

data_step1 = data_raw[:, mask_step1]

fids_step1_id = all_feature_id[mask_step1]
fids_step1_name = all_feature_name[mask_step1]


print("Step 1：丰度筛选完成")
print("-" * 60)
print(f"丰度阈值 P{ABUNDANCE_PERCENTILE}：{threshold_abundance:.6f}")
print(f"筛选前特征数：{data_raw.shape[1]}")
print(f"保留特征数：{data_step1.shape[1]}")
print(f"去除特征数：{int((~mask_step1).sum())}")

Step 1：丰度筛选完成
------------------------------------------------------------
丰度阈值 P40：3947.823479
筛选前特征数：1734
保留特征数：1040
去除特征数：694


In [5]:
# ============================================================
# 5. log1p 转换和 Step 2：IQR 筛选
# ============================================================

# 对丰度进行 log(1+x) 转换
log_step1 = np.log1p(data_step1)

q75 = np.percentile(log_step1, 75, axis=0)
q25 = np.percentile(log_step1, 25, axis=0)

iqr_values = q75 - q25

threshold_iqr = np.percentile(
    iqr_values,
    IQR_PERCENTILE,
)

# 保留 IQR 大于或等于 P25 的特征
mask_step2 = iqr_values >= threshold_iqr

if mask_step2.sum() == 0:
    raise ValueError(
        "IQR 筛选后没有剩余特征，请降低 IQR_PERCENTILE。"
    )

data_step2 = log_step1[:, mask_step2]

fids_step2_id = fids_step1_id[mask_step2]
fids_step2_name = fids_step1_name[mask_step2]

iqr_retained = iqr_values[mask_step2]
iqr_removed = iqr_values[~mask_step2]


print("Step 2：IQR 筛选完成")
print("-" * 60)
print(f"IQR 阈值 P{IQR_PERCENTILE}：{threshold_iqr:.6f}")
print(f"筛选前特征数：{data_step1.shape[1]}")
print(f"保留特征数：{data_step2.shape[1]}")
print(f"去除特征数：{int((~mask_step2).sum())}")
print(f"最终数据形状：{data_step2.shape}")

Step 2：IQR 筛选完成
------------------------------------------------------------
IQR 阈值 P25：1.324186
筛选前特征数：1040
保留特征数：780
去除特征数：260
最终数据形状：(159, 780)


In [6]:
# ============================================================
# 6. 生成不会重复的特征列名
# ============================================================

def make_unique_names(names):
    """
    将重复列名转换为唯一列名。

    例如：
    A, A, B
    转换为：
    A, A__2, B
    """
    counts = {}
    unique_names = []

    for name in names:
        name = str(name)

        if name not in counts:
            counts[name] = 1
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(
                f"{name}__{counts[name]}"
            )

    return unique_names


combined_feature_names = [
    f"{feature_id}_{feature_name}"
    for feature_id, feature_name in zip(
        fids_step2_id,
        fids_step2_name,
    )
]

combined_feature_names = make_unique_names(
    combined_feature_names
)

if len(set(combined_feature_names)) != len(combined_feature_names):
    raise RuntimeError(
        "处理后仍然存在重复特征列名。"
    )


print(f"最终特征列数：{len(combined_feature_names)}")
print("前 5 个特征列名：")

for name in combined_feature_names[:5]:
    print(f"  {name}")

最终特征列数：780
前 5 个特征列名：
  1_2-Butenal, (Z)-
  2_Unknown
  3_Unknown
  4_4,5-Dihydrooxazole-5-one, 4-[4-acetoxy-3-methoxybenzylidene]-2-phenyl-
  6_Cyclopropyl methyl carbinol


In [7]:
# ============================================================
# 7. 保存筛选后的 Excel 数据
# ============================================================

FILTERED_XLSX.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df_filtered = pd.DataFrame(
    data_step2,
    columns=combined_feature_names,
)

# Excel 中保留原始标签 1、2、3
df_filtered.insert(
    0,
    "Class",
    classes,
)

df_filtered.to_excel(
    FILTERED_XLSX,
    index=False,
)


print("筛选后的 Excel 已保存")
print("-" * 60)
print(f"保存位置：{FILTERED_XLSX.resolve()}")
print(f"样本数：{df_filtered.shape[0]}")
print(f"特征数：{df_filtered.shape[1] - 1}")

筛选后的 Excel 已保存
------------------------------------------------------------
保存位置：/home/liuxy/a-projects/BPD_jj/filtered_voc_step2.xlsx
样本数：159
特征数：780


In [8]:
# ============================================================
# 8. 保存特征筛选过程日志
# ============================================================

# 给 Step 1 之后的 IQR 值建立全特征映射
iqr_for_all_features = np.full(
    data_raw.shape[1],
    np.nan,
    dtype=np.float64,
)

iqr_for_all_features[mask_step1] = iqr_values


# 将 Step 2 的结果映射回全部特征
final_selected_mask = np.zeros(
    data_raw.shape[1],
    dtype=bool,
)

step1_indices = np.flatnonzero(mask_step1)

final_selected_indices = step1_indices[mask_step2]

final_selected_mask[final_selected_indices] = True


feature_log = pd.DataFrame({
    "feature_id": all_feature_id,
    "feature_name": all_feature_name,
    "mean_abundance": mean_data,
    "passed_abundance_filter": mask_step1,
    "iqr_after_log1p": iqr_for_all_features,
    "final_selected": final_selected_mask,
})

FEATURE_LOG_CSV.parent.mkdir(
    parents=True,
    exist_ok=True,
)

feature_log.to_csv(
    FEATURE_LOG_CSV,
    index=False,
    encoding="utf-8-sig",
)


print("特征筛选日志已保存")
print("-" * 60)
print(f"保存位置：{FEATURE_LOG_CSV.resolve()}")

特征筛选日志已保存
------------------------------------------------------------
保存位置：/home/liuxy/a-projects/BPD_jj/data/preprocessing_feature_log.csv


In [9]:
# ============================================================
# 9. 建立 PyTorch 数据集
# ============================================================

class VOCDataset(Dataset):
    def __init__(
        self,
        X,
        original_y,
        feature_names,
        feature_ids,
        metabolite_names,
        label_map=None,
    ):
        if label_map is None:
            label_map = {
                1: 0,
                2: 0,
                3: 1,
            }

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        original_y = np.asarray(
            original_y,
            dtype=np.int64,
        ).reshape(-1)

        if X.ndim != 2:
            raise ValueError(
                f"X 必须为二维数组，当前形状为：{X.shape}"
            )

        if len(original_y) != X.shape[0]:
            raise ValueError(
                "X 的样本数与标签数量不一致。"
            )

        mapped_y = pd.Series(
            original_y
        ).map(label_map)

        if mapped_y.isna().any():
            invalid_labels = sorted(
                pd.Series(original_y)[mapped_y.isna()]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"以下标签未在 label_map 中定义：{invalid_labels}"
            )

        mapped_y = mapped_y.to_numpy(
            dtype=np.int64,
        )

        self.X = torch.from_numpy(
            np.ascontiguousarray(X)
        )

        self.y = torch.from_numpy(
            np.ascontiguousarray(mapped_y)
        ).long()

        self.original_y = torch.from_numpy(
            np.ascontiguousarray(original_y)
        ).long()

        self.feature_names = list(feature_names)
        self.feature_ids = list(feature_ids)
        self.metabolite_names = list(metabolite_names)
        self.label_map = dict(label_map)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def to_mat(self, output_path):
        output_path = Path(output_path)

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        label_map_keys = np.asarray(
            list(self.label_map.keys()),
            dtype=np.int64,
        ).reshape(-1, 1)

        label_map_values = np.asarray(
            list(self.label_map.values()),
            dtype=np.int64,
        ).reshape(-1, 1)

        savemat(
            output_path,
            {
                # 模型输入
                "X": self.X.numpy().astype(np.float32),

                # 映射后的二分类标签：0、1
                "y": self.y.numpy()
                .astype(np.int64)
                .reshape(-1, 1),

                # 原始标签：1、2、3
                "original_y": self.original_y.numpy()
                .astype(np.int64)
                .reshape(-1, 1),

                # 特征信息
                "feat_names": np.asarray(
                    self.feature_names,
                    dtype=object,
                ).reshape(1, -1),

                "feature_ids": np.asarray(
                    self.feature_ids,
                    dtype=object,
                ).reshape(1, -1),

                "metabolite_names": np.asarray(
                    self.metabolite_names,
                    dtype=object,
                ).reshape(1, -1),

                # 标签映射
                "label_map_original": label_map_keys,
                "label_map_binary": label_map_values,
            },
        )

In [10]:
# ============================================================
# 10. 创建数据集并保存为 MAT 文件
# ============================================================

dataset = VOCDataset(
    X=data_step2,
    original_y=classes,
    feature_names=combined_feature_names,
    feature_ids=fids_step2_id,
    metabolite_names=fids_step2_name,
    label_map=LABEL_MAP,
)

dataset.to_mat(
    OUTPUT_MAT
)


class_counts = torch.bincount(
    dataset.y,
    minlength=2,
).tolist()


print("MAT 数据集已生成")
print("-" * 60)
print(f"保存位置：{OUTPUT_MAT.resolve()}")
print(f"样本数：{len(dataset)}")
print(f"特征数：{dataset.X.shape[1]}")
print(f"X 形状：{tuple(dataset.X.shape)}")
print(f"y 形状：{tuple(dataset.y.shape)}")
print(f"二分类标签数量：[类别0, 类别1] = {class_counts}")

MAT 数据集已生成
------------------------------------------------------------
保存位置：/home/liuxy/a-projects/BPD_jj/data/voc_dataset_1+2_vs_3.mat
样本数：159
特征数：780
X 形状：(159, 780)
y 形状：(159,)
二分类标签数量：[类别0, 类别1] = [106, 53]


In [11]:
# ============================================================
# 11. 重新读取 MAT 文件进行最终验证
# ============================================================

if not OUTPUT_MAT.exists():
    raise FileNotFoundError(
        f"MAT 文件保存失败：{OUTPUT_MAT.resolve()}"
    )

mat_check = loadmat(
    OUTPUT_MAT
)

X_check = np.asarray(
    mat_check["X"],
    dtype=np.float32,
)

y_check = np.asarray(
    mat_check["y"],
    dtype=np.int64,
).reshape(-1)


print("MAT 文件验证结果")
print("-" * 60)
print(f"X 形状：{X_check.shape}")
print(f"y 形状：{y_check.shape}")
print(f"标签及数量：{np.unique(y_check, return_counts=True)}")
print(f"是否包含 NaN：{bool(np.isnan(X_check).any())}")
print(f"是否包含 Inf：{bool(np.isinf(X_check).any())}")
print(f"数据最小值：{X_check.min():.6f}")
print(f"数据最大值：{X_check.max():.6f}")


if X_check.shape != tuple(dataset.X.shape):
    raise RuntimeError(
        "重新读取后的 X 形状与保存前不一致。"
    )

if y_check.shape[0] != len(dataset):
    raise RuntimeError(
        "重新读取后的标签数量与保存前不一致。"
    )

if not set(np.unique(y_check)).issubset({0, 1}):
    raise RuntimeError(
        f"最终标签不是 0/1：{np.unique(y_check)}"
    )


print("\n预处理全部完成。")

MAT 文件验证结果
------------------------------------------------------------
X 形状：(159, 780)
y 形状：(159,)
标签及数量：(array([0, 1]), array([106,  53]))
是否包含 NaN：False
是否包含 Inf：False
数据最小值：0.000000
数据最大值：18.205791

预处理全部完成。
